# Fraud Intelligence & Risk Analytics - Storytelling Notebook

## Business Context

This notebook analyzes transaction-level fraud intelligence data to answer key business questions:

- **What patterns indicate fraudulent transactions?**
- **How do risk features correlate with fraud outcomes?**
- **What are the characteristics of high-risk cards and merchants?**
- **How effective are our risk signals?**

## Dataset Overview

The `feat_transactions_risk` table contains:

- **Base transaction data**: amounts, timestamps, channels, countries
- **Velocity features**: transaction counts and amounts in 1h/24h windows
- **Behavioral features**: amount deviations, merchant diversity
- **Risk aggregates**: 30-day rolling fraud/decline rates for cards and merchants
- **Labels**: is_fraud (ground truth for model evaluation)

This is a **feature-rich, analytics-ready dataset** designed for:
- Exploratory data analysis
- Risk signal validation
- Business intelligence dashboards
- ML-lite model development

## Questions We Will Answer

1. **Fraud distribution**: What % of transactions are fraudulent? How does this vary by time, channel, country?
2. **Risk signal effectiveness**: Do high-risk cards/merchants actually have higher fraud rates?
3. **Feature importance**: Which features best distinguish fraud from legitimate transactions?
4. **Temporal patterns**: Are there time-of-day, day-of-week, or seasonal fraud patterns?
5. **Velocity analysis**: Do fraudsters show different transaction velocity patterns?

---

**Next Steps**: Load data, perform sanity checks, then dive into analysis.


In [1]:
# Import libraries
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine
import polars as pl
import plotly.graph_objects as go
import plotly.express as px

# Load environment variables
load_dotenv()

print("Libraries imported successfully")
print(f"Polars version: {pl.__version__}")


Libraries imported successfully
Polars version: 1.36.1


## Database Connection Setup

Connect to MySQL database using environment variables for credentials.


In [ ]:
# Build MySQL connection URL from environment variables
def get_mysql_url() -> str:
    """
    Build a SQLAlchemy MySQL connection URL from environment variables.
    
    Expected env vars:
      - MYSQL_HOST (default: localhost)
      - MYSQL_PORT (default: 3306)
      - MYSQL_USER (default: root)
      - MYSQL_PASSWORD (default: empty)
      - MYSQL_DB (default: fraud_db)
    """
    host = os.getenv("MYSQL_HOST", "localhost")
    port = os.getenv("MYSQL_PORT", "3306")
    user = os.getenv("MYSQL_USER", "root")
    password = os.getenv("MYSQL_PASSWORD", "sanalyst")  # Default password for local dev
    db = os.getenv("MYSQL_DB", "fraud_db")
    
    if password:
        return f"mysql+mysqlconnector://{user}:{password}@{host}:{port}/{db}"
    return f"mysql+mysqlconnector://{user}@{host}:{port}/{db}"


# Create SQLAlchemy engine
mysql_url = get_mysql_url()
engine = create_engine(mysql_url)

print("Database connection established")
print(f"Database: {os.getenv('MYSQL_DB', 'fraud_db')}")


Database connection established
Database: fraud_db


## Load Feature Table

Query `feat_transactions_risk` into a Polars DataFrame for fast analytics.


In [ ]:
# Query feat_transactions_risk table
query = """
SELECT *
FROM feat_transactions_risk
"""

try:
    # Load into Polars DataFrame
    df = pl.read_database(query, connection=engine.connect())
    
    print(f"Data loaded successfully")
    print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
except Exception as e:
    print(f"Error loading data: {e}")
    print(f"Error type: {type(e).__name__}")
    
    # Try alternative method: pandas -> polars
    print("\nTrying alternative method (pandas -> polars)...")
    try:
        import pandas as pd
        df_pd = pd.read_sql(query, engine)
        df = pl.from_pandas(df_pd)
        print(f"Data loaded successfully via pandas!")
        print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
    except Exception as e2:
        print(f"Alternative method also failed: {e2}")
        raise


ProgrammingError: (mysql.connector.errors.ProgrammingError) 1045 (28000): Access denied for user 'root'@'localhost' (using password: NO)
(Background on this error at: https://sqlalche.me/e/20/f405)

## Sanity Checks

Perform basic data quality checks to ensure the dataset is ready for analysis.


In [ ]:
# 1. Row count
print("=" * 60)
print("SANITY CHECKS")
print("=" * 60)
print(f"\n1. Row Count: {df.shape[0]:,} transactions")

# 2. Column list
print(f"\n2. Column Count: {df.shape[1]} columns")
print("\n   Key columns:")
key_columns = [
    'txn_id_clean', 'txn_ts', 'amount', 'is_fraud',
    'txns_last_24h', 'amt_last_24h', 'distinct_merchants_last_24h',
    'card_txn_count_30d', 'card_fraud_rate_30d',
    'merchant_txn_count_30d', 'merchant_fraud_rate_30d'
]
for col in key_columns:
    if col in df.columns:
        print(f"   ✓ {col}")
    else:
        print(f"   ✗ {col} (MISSING)")

# 3. Fraud vs non-fraud distribution
if 'is_fraud' in df.columns:
    fraud_counts = df.group_by('is_fraud').agg(pl.count().alias('count'))
    total = df.shape[0]
    
    print(f"\n3. Fraud Distribution:")
    for row in fraud_counts.iter_rows(named=True):
        fraud_label = "Fraud" if row['is_fraud'] == 1 else "Non-Fraud" if row['is_fraud'] == 0 else "Unknown"
        count = row['count']
        pct = (count / total * 100) if total > 0 else 0
        print(f"   {fraud_label}: {count:,} ({pct:.2f}%)")
else:
    print("\n3. Fraud Distribution: 'is_fraud' column not found")

# 4. Data types summary
print(f"\n4. Data Types Summary:")
dtype_counts = df.schema
print(f"   Numeric columns: {sum(1 for dt in dtype_counts.values() if dt in [pl.Int64, pl.Float64, pl.Decimal])}")
print(f"   String columns: {sum(1 for dt in dtype_counts.values() if dt == pl.Utf8)}")
print(f"   Boolean columns: {sum(1 for dt in dtype_counts.values() if dt == pl.Boolean)}")
print(f"   Date/Time columns: {sum(1 for dt in dtype_counts.values() if dt in [pl.Datetime, pl.Date])}")

# 5. Missing values check
print(f"\n5. Missing Values (top 10 columns):")
try:
    null_counts = df.null_count()
    # Convert to long format and sort
    null_df = null_counts.melt(variable_name='column', value_name='null_count')
    null_df = null_df.with_columns([
        (pl.col('null_count') / df.shape[0] * 100).alias('null_pct')
    ]).sort('null_count', descending=True).head(10)

    has_nulls = False
    for row in null_df.iter_rows(named=True):
        col_name = row['column']
        null_count = row['null_count']
        null_pct_val = row['null_pct']
        if null_count > 0:
            print(f"   {col_name}: {null_count:,} ({null_pct_val:.2f}%)")
            has_nulls = True
    
    if not has_nulls:
        print("   No missing values found in top 10 columns")
except Exception as e:
    print(f"   Error checking nulls: {e}")
    # Fallback: simple null count per column
    print("   Using simple null count...")
    for col in df.columns[:10]:  # First 10 columns
        null_count = df[col].null_count()
        if null_count > 0:
            pct = (null_count / df.shape[0] * 100) if df.shape[0] > 0 else 0
            print(f"   {col}: {null_count:,} ({pct:.2f}%)")

print("\n" + "=" * 60)
print("Sanity checks complete!")
print("=" * 60)


## Dataset Summary

Quick summary of the loaded dataset structure.


In [ ]:
# Display basic info
print("Dataset Info:")
print(f"  Rows: {df.shape[0]:,}")
print(f"  Columns: {df.shape[1]}")
print(f"  Memory usage: {df.estimated_size('mb'):.2f} MB")

# Show first few rows
print("\nFirst 5 rows:")
df.head(5)


## Next Steps

Now that data is loaded and validated, we can proceed with:

1. **Exploratory Data Analysis**: Visualize fraud distribution, feature distributions
2. **Risk Signal Analysis**: Validate if high-risk cards/merchants correlate with fraud
3. **Feature Analysis**: Understand which features are most predictive
4. **Temporal Analysis**: Identify time-based fraud patterns
5. **Business Insights**: Generate actionable recommendations

---

**Ready for analysis!** 🚀
